# 🧪 Sesión 38 · Tareas en Segundo Plano (Background Tasks) Asíncronas

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Cómo respondes rápido cuando la tarea es lenta?

---

## 🖥️ Así se verá el resultado final

```http
POST /api/reportes  {"email":"ana@ejemplo.com"}
HTTP/1.1 202 Accepted
{"tarea_id":"r1","estado":"en_proceso"}

GET /api/reportes/r1
HTTP/1.1 200 OK
{"tarea_id":"r1","estado":"completado"}

$ cat reportes.log
ana@ejemplo.com
```

_El cliente recibe respuesta antes de que termine el trabajo; el log demuestra que sí se ejecutó._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Responder 202 antes de terminar el trabajo pesado.
- Delegar el trabajo con BackgroundTasks.
- Verificar el estado de la tarea y su efecto secundario.

## ✅ Criterios de aceptación

El verificador ejecutará los casos HTTP y comprobará los archivos del contrato. Todo debe quedar en verde:

- [ ] El reporte se acepta de inmediato → HTTP 202
- [ ] La tarea se consulta por id → HTTP 200
- [ ] Una tarea inexistente es 404 → HTTP 404
- [ ] El trabajo quedó registrado en el log
- [ ] Se usa BackgroundTasks

## 🧩 El reto

Implementa un reporte que se procese en segundo plano y pueda consultarse por su identificador.

**Requisitos del contrato**

- POST /api/reportes responde 202 con tarea_id y estado en_proceso.
- El trabajo se agrega con BackgroundTasks y escribe el email en reportes.log.
- GET /api/reportes/{tarea_id} responde 200 con estado completado.
- Una tarea inexistente responde 404.
- El log existe después de ejecutar la petición.

> **Pista conceptual:** `BackgroundTasks` se inyecta como parámetro y se ejecuta después de la respuesta. No hagas el trabajo pesado antes de responder.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion38.md` con el tiempo de respuesta y el contenido del log.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [ ]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/main/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [ ]:
%%writefile app.py
from fastapi import FastAPI

app = FastAPI(title="Tu tarea en segundo plano se verá así")

# Implementa aquí el contrato declarado en el reto.
# Tú decides los modelos, las rutas, las validaciones y el manejo de errores.


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Framework, versión y restricciones del entorno. |
| **Rol** | Qué papel técnico debe asumir el modelo. |
| **Especificación** | Rutas, entradas, salidas, validaciones y códigos HTTP. |
| **Output** | Qué formato exacto esperas de vuelta. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [ ]:
from verificar import probar_http

casos = [{'descripcion': 'El reporte se acepta de inmediato',
  'metodo': 'POST',
  'ruta': '/api/reportes',
  'json': {'email': 'ana@ejemplo.com'},
  'estado': 202,
  'json_contiene': {'estado': 'en_proceso'},
  'guardar': {'tarea_id': 'tarea_id'}},
 {'descripcion': 'La tarea se consulta por id',
  'metodo': 'GET',
  'ruta': '/api/reportes/{tarea_id}',
  'estado': 200,
  'json_contiene': {'estado': 'completado'}},
 {'descripcion': 'Una tarea inexistente es 404', 'metodo': 'GET', 'ruta': '/api/reportes/no-existe', 'estado': 404}]

probar_http(casos, archivo="app.py", limpiar=['reportes.log'], env={})


In [ ]:
from verificar import revisar_python, resumen

resultados = []
resultados.append(revisar_python("from pathlib import Path\ntexto = Path(\"reportes.log\").read_text(encoding=\"utf-8\")\nassert \"ana@ejemplo.com\" in texto, texto\n", "El trabajo quedó registrado en el log"))
resumen(resultados)


In [ ]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("app.py", exigir=["BackgroundTasks"], descripcion="Se usa BackgroundTasks"))
resumen(resultados)


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion38.md` a `auditorias/auditoria_sesion38.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
